# SSURGO Soil Data Cleaning

**Objective:** Build one soil polygon layer for the Salinas River watershed (HUC8 18060005) with the SSURGO attributes used as model features: slope, depth to bedrock, available water storage, drainage class, and hydrologic soil group.

**Steps**:

1. Soil polygons clipped to the watershed area

2. Assign column names

3. Select soil attributes

4. Join soil attributes to polygons and check result

5. Save files



**Inputs (local only, not tracked by Git)**:

- SSURGO survey downloads (one folder per survey area), each with `spatial/soilmu_a_*.shp` and `tabular/comp.txt`, `muaggatt.txt`, `mstabcol.txt`

- NHD HU8 geodatabase `NHD_H_18060005_HU8_GDB.gdb` (layer `WBDHU8`), used as the watershed boundary

**Outputs (local only):** 

- `data/soils/soils_huc8_attributes.gpkg`, one row per soil polygon with its map-unit attributes. It is read by `notebooks/merge/all_merge_final.ipynb`, which computes area-weighted features within each well buffer.


In [19]:
from pathlib import Path
import sys

import pandas as pd
import geopandas as gpd

# Make src importable from any notebook folder (identical in every notebook)
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())))
from src import paths

PROJECT_ROOT = paths.ROOT

# ===== Inputs (raw downloads, not in git; see the README's data sources) =====
SOIL_FOLDER = paths.RAW_SSURGO      # folder that holds the CA### SSURGO survey folders
HUC8_GDB = paths.HUC8_GDB           # USGS NHD geodatabase for HU8 18060005

# ===== Output =====
OUTPUT_PATH = paths.SOILS

paths.require(SOIL_FOLDER, HUC8_GDB)
#print("SSURGO folder:", SOIL_FOLDER)
#print("HUC8 geodatabase:", HUC8_GDB)
#print("Output:", OUTPUT_PATH)

from src.data.soils import clip_to_watershed, column_names, soil_attributes, join_attributes_to_polygons

## 1. Soil polygons clipped to the watershed

Read every SSURGO survey's map-unit polygons, combine them, and keep only the area inside the Salinas HUC8 boundary.

In [20]:
soils_huc8 = clip_to_watershed(SOIL_FOLDER, HUC8_GDB)

Find and combine all SSURGO soil polygon shapefiles.
5 survey areas in the raw data with 27,653 polygons

Clip soils to the watershed:
Dataframe reduced to (15906, 5) out of (27653, 5)


## 2. Assign column names

The tabular raw files for the soil data have no headers, so we add in column names from the SSURGO metadata table `mstabcol.txt`, which is identical in every survey. <!-- Only map units inside the watershed are kept.-->

In [21]:
muaggatt_huc8 = column_names(soils_huc8, SOIL_FOLDER)

Column names from metadata:
Index(['comppct_l', 'comppct_r', 'comppct_h', 'compname', 'compkind',
       'majcompflag', 'otherph', 'localphase', 'slope_l', 'slope_r',
       ...
       'flsoilleachpot', 'flsoirunoffpot', 'fltemik2use', 'fltriumph2use',
       'indraingrp', 'innitrateleachi', 'misoimgmtgrp', 'vasoimgtgrp', 'mukey',
       'cokey'],
      dtype='str', length=109)

Get only the MUKEY values that are inside the HUC8 watershed.
Dataframe reduced to size:
(3074, 109)

Find, read and combine the map-unit aggregate tables, creating dataframe of size:
(492, 40)



## 3. Select soil attributes

Candidate attributes related to nitrate leaching and hydrology are selected first, then checked for missing values at the map-unit level. Water-table depth and flooding frequency were dropped after the coverage check; the five attributes kept are listed below.

In [22]:
soils_attributes = soil_attributes(muaggatt_huc8)

Select soil variables relevant to nitrate transport and hydrology:


,mukey,slopegradwta,brockdepmin,wtdepannmin,aws0150wta,drclassdcd,hydgrpdcd,flodfreqdcd
0,455537,6.0,91.0,NaN,14.56,Well drained,D,NaN
1,455538,12.0,91.0,NaN,14.56,Well drained,D,NaN
2,455539,23.0,91.0,NaN,14.56,Well drained,D,NaN
3,455540,40.0,91.0,NaN,14.56,Well drained,D,NaN
4,455541,38.0,43.0,NaN,12.00,Well drained,D,NaN



Keep soil predictors with good data coverage:


,mukey,slopegradwta,brockdepmin,aws0150wta,drclassdcd,hydgrpdcd
0,455537,6.0,91.0,14.56,Well drained,D
1,455538,12.0,91.0,14.56,Well drained,D
2,455539,23.0,91.0,14.56,Well drained,D
3,455540,40.0,91.0,14.56,Well drained,D
4,455541,38.0,43.0,12.00,Well drained,D



mukey = unique ID for each SSURGO soil map unit
slopegradwta = weighted average slope gradient for the map unit
brockdepmin = minimum depth to bedrock within the map unit
aws0150wta = weighted average available water storage in the top 150 cm of soil
drclassdcd = dominant drainage class of the map unit
hydgrpdcd = dominant hydrologic soil group of the map unit



## 4. Join attributes to polygons and check the result

In [23]:
soils_model = join_attributes_to_polygons(soils_huc8, soils_attributes)

Check missing values at the unique soil map-unit level.

Missing values:


slopegradwta      1
brockdepmin     234
aws0150wta       16
drclassdcd       31
hydgrpdcd        23
dtype: int64


Check for duplicated rows and invalid geometries (both should be 0)
Duplicated rows: 0
Invalid geometries: 0



## 5. Save

In [24]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
soils_model.to_file(OUTPUT_PATH, layer="soils_huc8_attributes", driver="GPKG")
print(f"Saved {len(soils_model):,} polygons to {OUTPUT_PATH.relative_to(PROJECT_ROOT)}")

Saved 15,906 polygons to data\soils\soils_huc8_attributes.gpkg


## 6. Conclusions

- The watershed is covered by **5 SSURGO survey areas**: **15,906 soil polygons** belonging to **492 map units**.
- Five attributes are kept: slope (`slopegradwta`), minimum depth to bedrock (`brockdepmin`), available water storage 0–150 cm (`aws0150wta`), drainage class (`drclassdcd`), and hydrologic soil group (`hydgrpdcd`).
- Missing values at the map-unit level (of 492): depth to bedrock 234, drainage class 31, hydrologic group 23, water storage 16, slope 1. Depth to bedrock is later removed in feature selection (73% missing at the well level).
- No duplicated rows or invalid geometries.
- Polygon attributes are converted to well-level, area-weighted features (500 m buffers) in `notebooks/data_merge/all_merge_final.ipynb`.